# Explain the remaining cassette-front misses

**CPU only. Run all four code cells. No training, model download or image
download is needed.** This notebook reuses the earlier full decoded
prediction archive, about 15.6 MB, and the completed three-reference
comparison. It checks that they describe the same checkpoint and images.

The first analysis traces the current missed fronts through confidence
filtering, ordinary suppression, clipping, final suppression and reference
matching. The second describes box size and extent disagreement. It does
not lower the evaluation threshold, repair labels or choose a new model.

Pipeline stages describe where eligible candidates disappear. They do not
establish why training produced a particular score or box. A nearest box
selected using reference overlap is a diagnostic association, not proof
that both boxes depict the same physical front.


## 1 Load the published implementation

Source, downloaded inputs and generated outputs stay separate. A changed
source directory is preserved; choose a fresh `BAOMAI_PROPOSAL_AUDIT_BASE`
to start another run.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "bb1d74cce4739a72ad0f7d9f227ca1e4cac37c7c"
DRIVER_HASHES = {'analyze_proposal_stages_r63.py': '0dd585cb7d957dcd599c0f40fbbf8fcc1ffc85c9bf98f98307eb0c799f29f15b', 'analyze_box_geometry_r63.py': 'a6e5dfcba4b023ccefca56ca3ed3743b7ac704b139c0b469b896079783e73618'}
BASE = Path(os.environ.get("BAOMAI_PROPOSAL_AUDIT_BASE", "/content/baomai-proposal-audit-r63")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
CACHE = Path(os.environ.get("BAOMAI_PROPOSAL_AUDIT_CACHE", str(BASE / "cache"))).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Existing source preserved; choose a fresh BAOMAI_PROPOSAL_AUDIT_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "/scripts/", "/docs/thesis/evidence/annotation-reference/baomai-annotation-reference-r60-results.zip"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
def verify_source():
    if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
        raise RuntimeError("Changed source preserved; choose a fresh BAOMAI_PROPOSAL_AUDIT_BASE.")
    for name, expected in DRIVER_HASHES.items():
        if hashlib.sha256((SOURCE / "scripts" / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Source identity differs: " + name)
verify_source()
WORK.mkdir(parents=True, exist_ok=True)
print("Published source:", SOURCE_COMMIT)


hint: Using 'master' as the name for the initial branch. This default branch name
hint: will change to "main" in Git 3.0. To configure the initial branch name
hint: to use in all of your new repositories, which will suppress this warning,
hint: call:
hint:
hint: 	git config --global init.defaultBranch <name>
hint:
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint:
hint: 	git branch -m <name>
hint:
hint: Disable this message with "git config set advice.defaultBranchName false"
Initialized empty Git repository in /workspace/scratch/f5388ec04e55/r63-notebook-run/source/.git/
From https://github.com/Siuuuers/dl
 * branch            bb1d74cce4739a72ad0f7d9f227ca1e4cac37c7c -> FETCH_HEAD
HEAD is now at bb1d74c Add reproducible proposal-stage and box-geometry diagnostics
Published source: bb1d74cce4739a72ad0f7d9f227ca1e4cac37c7c


## 2 Obtain the verified decoded predictions

NumPy is the only numerical dependency. The earlier archive contains all
decoded proposals for the same 20 photographs, before score filtering
and suppression. Its byte count and SHA-256 must match the frozen return.
Existing compatible downloads are reused.


In [2]:
import importlib.util
if importlib.util.find_spec("numpy") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "numpy==2.4.6"])
import numpy as np
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE / "scripts"))
from audit_front_diagnosis_r49_results import obtain_archive, ARCHIVE_BYTES, ARCHIVE_SHA
R49_ARCHIVE = obtain_archive(CACHE)
assert R49_ARCHIVE.stat().st_size == ARCHIVE_BYTES
assert hashlib.sha256(R49_ARCHIVE.read_bytes()).hexdigest() == ARCHIVE_SHA
REFERENCE_ARCHIVE = SOURCE / "docs/thesis/evidence/annotation-reference/baomai-annotation-reference-r60-results.zip"
print("Decoded archive verified:", R49_ARCHIVE.name, ARCHIVE_BYTES, "bytes")
print("NumPy:", np.__version__)


Decoded archive verified: baomai-r49-diagnosis-results.zip 15631804 bytes
NumPy: 2.3.5


## 3 Trace the current errors and inspect box geometry

Historical native outputs must replay exactly. At the current operating
point, matching and scored outcomes must agree with the completed
comparison, while any CPU/GPU numerical differences are recorded.

The main detection confidence remains strictly above 0.25, native and
final NMS overlap 0.70, and reference matching IoU at least 0.50. The
existing two-view corner results remain separate: the older full tensors
describe the original-view boxes used by that method.


In [3]:
STAGES, GEOMETRY = WORK / "proposal-stages", WORK / "box-geometry"
def stage_command(output):
    return [sys.executable, "-B", SOURCE / "scripts/analyze_proposal_stages_r63.py",
            "--root", SOURCE, "--r49-archive", R49_ARCHIVE,
            "--reference-archive", REFERENCE_ARCHIVE, "--output", output]
def geometry_command(output):
    return [sys.executable, "-B", SOURCE / "scripts/analyze_box_geometry_r63.py",
            "--root", SOURCE, "--output", output]
run(stage_command(STAGES))
run(geometry_command(GEOMETRY))
from IPython.display import Markdown, display
display(Markdown((STAGES / "results.md").read_text()))
display(Markdown((GEOMETRY / "results.md").read_text()))


{
  "current_miss_stages": {
    "confidence_gate": 12,
    "no_iou_eligible_decoded_box": 3,
    "ordinary_nms": 3,
    "postclip_suppression": 0,
    "retained_assignment_contention": 0
  },
  "files": {
    "cases.json": {
      "bytes": 67683,
      "sha256": "ebcc8122ed685bba15add19558bc12dbcc49bbfafbb01718de6f7e9ec0ccdc49"
    },
    "results.md": {
      "bytes": 5338,
      "sha256": "0d58b6b043aefa85abe015dfd2b7f18f228600b722bf816b3fbfb9a07f0354e1"
    },
    "summary.json": {
      "bytes": 77328,
      "sha256": "82dbd244d78c4a9b6066d9bb17764cbebd5468d438db22328e1523dc718730c3"
    }
  },
  "status": "completed"
}
{"files": ["cases.json", "manifest.json", "results.md", "summary.json"], "status": "completed", "verification": {"canonical_predictions_exactly_reconstructed": true, "independent_distance_identity_checked": true, "missed_case_records": 197, "per_image_score_tables_exactly_reproduced": 120, "reference_targets_analyzed_per_method": 1984}}


# Current missed-front proposal stages

This CPU replay explains the 18 missed reviewed fronts of the retained method at confidence > 0.25. It changes no labels, outputs, or model weights.

| Stage where IoU-eligible support disappears | Missed fronts |
|---|---:|
| no iou eligible decoded box | 3 |
| confidence gate | 12 |
| ordinary nms | 3 |
| postclip suppression | 0 |
| retained assignment contention | 0 |

All 18 missed fronts have incomplete corner references: 13 have one available corner and five have none. The retained measurement remains 636/654 matched fronts, 51 extra predictions including five duplicates, and 569/583 complete pairs within five pixels.

## Continuity checks

Both hash-pinned archives and every inventoried member were verified. The archived model and photograph hashes agree. All 20 historical 0.13 native outputs, native anchor indices, and postclip outputs replay exactly. At 0.25, all 20 current native record orders and postclip identities agree; all 60 reference-image comparisons preserve matching, missed/extra/duplicate identities, and corner threshold outcomes.

The historical tensors were generated on a Tesla T4; the current cache was generated on CPU. The maximum coordinate difference is 0.00018310546875 pixels, and the maximum box or corner score difference is 3.75509262085e-06. These are measured agreement checks, not byte identity.

## Geometric support and competition

| Candidate stage | Candidates | References with support | Maximum one-to-one matches | Candidates eligible for multiple references |
|---|---:|---:|---:|---:|
| decoded | 386400 | 651 | 651 | 9 |
| above confidence | 6438 | 639 | 639 | 0 |
| native nms | 693 | 636 | 636 | 0 |
| postclip | 687 | 636 | 636 | 0 |

A reference with one eligible proposal is not necessarily independently recoverable. The one-to-one bounds account for shared candidates; the actual scorer also retains its original greedy assignment. Multiple raw proposals are not automatically multiple physical fronts or duplicate errors. The detailed cases include every above-threshold eligible proposal lost to NMS and its actual suppressor. The existing extra postclip pass removes six duplicate predictions: native outputs have 57 extras including 11 duplicates; postclip outputs have 51 extras including five duplicates. Both stages match the same 636 references.

## Every current missed front

| Photograph / reviewed row | Available corners | Stage | Best decoded IoU | Highest eligible score | Matched at historical 0.13 |
|---|---:|---|---:|---:|---|
| new_000008 / 1 | 1 | confidence gate | 0.918381 | 0.090770 | no |
| new_000008 / 33 | 1 | confidence gate | 0.926685 | 0.011005 | no |
| new_000008 / 34 | 1 | confidence gate | 0.839737 | 0.072173 | no |
| new_000009 / 51 | 1 | no iou eligible decoded box | 0.490614 | none | no |
| new_000009 / 54 | 0 | confidence gate | 0.765790 | 0.023850 | no |
| new_000010 / 14 | 0 | confidence gate | 0.842348 | 0.146784 | yes |
| new_000010 / 30 | 1 | confidence gate | 0.779465 | 0.058395 | no |
| new_000010 / 32 | 0 | confidence gate | 0.839197 | 0.167130 | yes |
| new_000010 / 33 | 0 | confidence gate | 0.904613 | 0.107094 | no |
| new_000011 / 55 | 1 | confidence gate | 0.630831 | 0.003410 | no |
| new_000011 / 56 | 1 | confidence gate | 0.571046 | 0.010447 | no |
| new_000011 / 57 | 1 | no iou eligible decoded box | 0.306544 | none | no |
| new_000011 / 59 | 1 | confidence gate | 0.750754 | 0.092318 | no |
| new_000011 / 60 | 0 | no iou eligible decoded box | 0.402694 | none | no |
| new_000088 / 51 | 1 | confidence gate | 0.657288 | 0.017721 | no |
| new_000089 / 43 | 1 | ordinary nms | 0.625526 | 0.490754 | yes |
| new_000089 / 48 | 1 | ordinary nms | 0.734533 | 0.795476 | no |
| new_000090 / 39 | 1 | ordinary nms | 0.929821 | 0.691672 | no |

## Interpretation limits

- Exploratory diagnosis of 20 repeatedly used development photographs, not independent test evidence.
- The two archives record identical model and photograph SHA-256 values; this replay does not load model weights or photograph bytes.
- Decoded tensors came from the historical Tesla T4 runtime. Current retained outputs came from a newer CPU execution with different Python and PyTorch versions. Numeric bytes are not identical; measured identities and threshold outcomes are verified separately.
- Only the original-view trained checkpoint has these decoded tensors. Reflected proposals and future newly trained checkpoints are not diagnosed.
- IoU eligibility is a geometric relation to the reviewed annotations. Many eligible proposals can describe the same physical object, and one proposal can overlap multiple references.
- Maximum one-to-one matching is a diagnostic upper bound for each fixed candidate set, not a measured detector result or a deployable oracle.
- A stage label identifies where eligible support disappears in this fixed computation. It does not prove an optimal intervention, correct physical annotations, or recoverable performance without extra predictions.
- No eligible decoded box means no box reaches this reference IoU threshold in the finite decoded tensor; it does not mean the neural network had no visual response.
- The audit does not change the current confidence threshold, suppression policy, labels, retained method, or two-view corner measurements.


# Box geometry diagnostic

Saved predictions and all three annotation references are unchanged. These are descriptive development-data measurements.

| Reference | Method | Missed | Near-overlap misses | Candidate width >1.5x | Candidate height >1.5x |
| --- | --- | ---: | ---: | ---: | ---: |
| supplied664 | original | 39 | 20 | 1 | 12 |
| supplied664 | proposed | 46 | 29 | 1 | 16 |
| frozen666 | original | 26 | 6 | 2 | 3 |
| frozen666 | proposed | 34 | 17 | 0 | 7 |
| r50 | original | 34 | 19 | 15 | 4 |
| r50 | proposed | 18 | 10 | 8 | 1 |

## Reviewed-reference geometry of the retained method

| Group | Fronts | Median width (px) | Median height (px) | Median square-root area (px) |
| --- | ---: | ---: | ---: | ---: |
| all | 654 | 92.000 | 28.995 | 50.927 |
| complete | 583 | 93.120 | 29.000 | 51.846 |
| incomplete_matched | 53 | 81.000 | 24.000 | 42.790 |
| incomplete_missed | 18 | 35.500 | 23.005 | 28.853 |
| incomplete_near_overlap_missed | 10 | 31.500 | 18.500 | 25.229 |

## The ten near-overlap misses under the reviewed reference

| Case | Reference width x height (px) | Max IoU | Width ratio | Height ratio | Signed left / top / right / bottom errors (px) | W2 distance (px) |
| --- | --- | ---: | ---: | ---: | --- | ---: |
| r50/new_000008/R001 | 217.50 x 10.00 | 0.3906 | 0.992 | 2.490 | +5.48 / +0.00 / +3.76 / +14.90 | 11.538 |
| r50/new_000009/R051 | 21.00 x 16.00 | 0.4906 | 1.308 | 0.930 | -5.59 / +4.33 / +0.87 / +3.21 | 5.526 |
| r50/new_000011/R055 | 31.00 x 18.00 | 0.4333 | 2.080 | 1.109 | -1.51 / -0.64 / +31.99 / +1.33 | 22.666 |
| r50/new_000011/R056 | 20.00 x 16.00 | 0.3723 | 2.192 | 1.220 | -1.95 / +0.05 / +21.89 / +3.57 | 15.744 |
| r50/new_000011/R057 | 15.50 x 12.00 | 0.3065 | 2.300 | 1.320 | -5.18 / +0.64 / +14.96 / +4.48 | 11.644 |
| r50/new_000011/R059 | 32.00 x 24.00 | 0.4726 | 1.777 | 1.183 | -25.01 / -1.37 / -0.15 / +3.02 | 17.839 |
| r50/new_000088/R051 | 28.50 x 23.01 | 0.4293 | 2.126 | 1.094 | -29.15 / +0.02 / +2.95 / +2.19 | 20.775 |
| r50/new_000089/R043 | 32.50 x 19.00 | 0.4689 | 1.813 | 1.176 | -0.76 / -0.70 / +25.68 / +2.65 | 18.269 |
| r50/new_000089/R048 | 46.00 x 32.00 | 0.4951 | 1.838 | 1.099 | -36.14 / -1.28 / +2.43 / +1.88 | 25.662 |
| r50/new_000090/R039 | 51.00 x 21.00 | 0.4937 | 1.517 | 1.335 | -0.57 / -5.05 / +25.79 / +1.99 | 18.642 |

## Interpretation

A near-overlap miss is an unmatched reference whose highest retained-prediction IoU is at least 0.30 and below 0.50. The diagnostic does not change the official one-to-one assignment. It may reuse a prediction or select a box already assigned to a different reference; cases.json records that status.

The overlap-selected candidate is not verified as the same physical front. All-zero-overlap cases have no selected candidate. Width/height ratios and signed edge errors indicate what to inspect, not which label or prediction should be changed.

The pixel distance is the 2-Wasserstein distance between Gaussian box representations with mean (center_x, center_y) and diagonal covariance (width^2/4, height^2/4). No normalization constant is selected and no distance replaces IoU in evaluation. A narrow visible fragment is not necessarily a tiny full object.

These measurements motivate an audit of visible extent, occlusion ownership and training augmentation before another loss-function experiment. They do not establish the training-time cause or a model improvement.


## 4 Recompute, verify and download

Both analyses run again into separate verification directories. Every
expected file must match the first run byte for byte. The result ZIP
contains only the verified files and a manifest. These results explain
development errors; they are not a new trained-model result or a robot
positioning test.


In [4]:
from zipfile import ZipFile, ZipInfo, ZIP_DEFLATED
verify_source()
REPLAY = WORK / "verification"
run(stage_command(REPLAY / "proposal-stages"))
run(geometry_command(REPLAY / "box-geometry"))
EXPECTED = ('summary.json', 'cases.json', 'results.md', 'manifest.json')
entries = {}
for name, directory in (("proposal-stages", STAGES), ("box-geometry", GEOMETRY)):
    for filename in EXPECTED:
        original = (directory / filename).read_bytes()
        replayed = (REPLAY / name / filename).read_bytes()
        if original != replayed:
            raise RuntimeError("Saved result differs on replay: " + name + "/" + filename)
        entries[name + "/" + filename] = original
manifest = {"schema": "cassette-proposal-notebook-package-r63-v1",
            "source_commit": SOURCE_COMMIT, "driver_hashes": DRIVER_HASHES,
            "fresh_inference": False, "training": False,
            "files": {name: {"bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()}
                      for name, data in sorted(entries.items())}}
entries["manifest.json"] = (json.dumps(manifest, indent=2, sort_keys=True) + "\n").encode()
RETURN = WORK / "baomai-proposal-audit-r63-results.zip"
temporary = WORK / "baomai-proposal-audit-r63-results.tmp.zip"
with ZipFile(temporary, "w", compression=ZIP_DEFLATED, compresslevel=9) as bundle:
    for name, data in sorted(entries.items()):
        info = ZipInfo(name, (2026, 1, 1, 0, 0, 0))
        info.compress_type = ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        bundle.writestr(info, data, compress_type=ZIP_DEFLATED, compresslevel=9)
with ZipFile(temporary) as bundle:
    assert bundle.testzip() is None and set(bundle.namelist()) == set(entries)
    assert len(bundle.namelist()) == len(entries)
    for name, data in entries.items():
        assert bundle.read(name) == data
if RETURN.exists() and RETURN.read_bytes() != temporary.read_bytes():
    raise RuntimeError("Different existing package preserved; choose a fresh base directory.")
temporary.replace(RETURN)
verify_source()
print("Both analyses replayed exactly; verified", len(entries), "ZIP members.")
from IPython.display import FileLink
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RETURN)))
else:
    files.download(str(RETURN))


{
  "current_miss_stages": {
    "confidence_gate": 12,
    "no_iou_eligible_decoded_box": 3,
    "ordinary_nms": 3,
    "postclip_suppression": 0,
    "retained_assignment_contention": 0
  },
  "files": {
    "cases.json": {
      "bytes": 67683,
      "sha256": "ebcc8122ed685bba15add19558bc12dbcc49bbfafbb01718de6f7e9ec0ccdc49"
    },
    "results.md": {
      "bytes": 5338,
      "sha256": "0d58b6b043aefa85abe015dfd2b7f18f228600b722bf816b3fbfb9a07f0354e1"
    },
    "summary.json": {
      "bytes": 77328,
      "sha256": "82dbd244d78c4a9b6066d9bb17764cbebd5468d438db22328e1523dc718730c3"
    }
  },
  "status": "completed"
}
{"files": ["cases.json", "manifest.json", "results.md", "summary.json"], "status": "completed", "verification": {"canonical_predictions_exactly_reconstructed": true, "independent_distance_identity_checked": true, "missed_case_records": 197, "per_image_score_tables_exactly_reproduced": 120, "reference_targets_analyzed_per_method": 1984}}
Both analyses replayed exact

/workspace/scratch/f5388ec04e55/r63-notebook-run/work/baomai-proposal-audit-r63-results.zip